# Etapa 1: Introdução e Análise Inicial do Sinal ECG (MIT BIH)
## Projeto de Processamento de Sinais Digitais (Grupo 6)

> Pedro Arthur, RA: 814248

> Juliano Eleno Silva Pádua, RA: 800812

> Matheo, RA: 821293

**Registo de referência:** MIT BIH 100 (ritmo sinusal normal)

* Esta apresentação cobre a Etapa 01 do projeto.
* Objetivo: remover ruídos típicos de ECG.
* Validação estritamente visual sobre os 10 segundos iniciais do Registo 100.

## 1. Resumo do Projeto e Objetivos

**Problema.** Sinais reais de ECG carregam três famílias de ruído que comprometem o diagnóstico:

* Baseline wander (abaixo de 0.5 Hz), associado à respiração e ao movimento do paciente.
* Interferência da rede elétrica (PLI), próxima de 60 Hz, decorrente do acoplamento à rede de energia.
* Ruído muscular ou EMG (acima de 40 Hz), originado por contrações musculares involuntárias.

**Objetivos da Fase 1.**

* Atenuar as três famílias de ruído com filtros FIR de fase linear.
* Preservar a morfologia das ondas P, do complexo QRS e da onda T, evitando deslocamento temporal.
* Realizar uma comparação visual, no domínio do tempo, entre dois métodos de projeto FIR aplicados ao sinal de referência.

## 2. Anatomia do ECG: Ondas P, QRS e T

![Anatomia do ECG](../imgs/image.png)

Cada batimento cardíaco gera um padrão fisiológico bem definido no ECG:

* Onda P: despolarização (ativação elétrica) dos átrios. Pequena amplitude e baixa frequência.
* Complexo QRS: despolarização dos ventrículos. É o pico de maior amplitude e a marca temporal do batimento (pico R).
* Onda T: repolarização ventricular, ou seja, a recuperação elétrica do coração antes do próximo ciclo.

**Justificativa do filtro FIR de fase linear.**

* Atraso de grupo constante: todos os componentes de frequência são deslocados pelo mesmo tempo.
* Resultado: as ondas P, QRS e T não são deformadas nem deslocadas relativamente entre si.
* Trata se de uma exigência clínica, pois medir intervalos PR, QRS e QT requer preservação morfológica.

## 3. Base de Dados: MIT BIH Arrhythmia Database

* Referência mundial em cardiologia computacional, distribuída pela PhysioNet.
* 48 registos de ECG de pacientes reais, com aproximadamente 30 minutos cada e dois canais (tipicamente MLII e V5).
* Frequência de amostragem: `fs = 360 Hz` (Nyquist em 180 Hz, suficiente para a faixa diagnóstica entre 0.5 Hz e 40 Hz).
* Anotações `.atr` produzidas por especialistas, identificando o instante do pico R e o tipo de batimento.
* Registo 100: escolhido para a Fase 1 por apresentar ritmo sinusal normal, servindo de referência de normalidade para validar que o filtro não introduz artefatos.

In [ ]:
from __future__ import annotations
from pathlib import Path
import scipy.signal as sp_signal
import matplotlib.axes
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import wfdb

from src.config import mitdb_record_dir
from src.preprocessing.fir_filters import (
    design_highpass,
    design_highpass_remez,
    design_bandstop,
    design_bandstop_remez,
    design_lowpass,
    design_lowpass_remez,
    apply_filtfilt,
)

RECORD_NAME = "100"
record_path = (Path(mitdb_record_dir()) / RECORD_NAME).as_posix()

record = wfdb.rdrecord(record_path)
ann = wfdb.rdann(record_path, "atr")

print("fs (Hz):", record.fs)
print("Canais:", record.sig_name)
print("Duração (s):", record.sig_len / record.fs)

## 4. Visualização do Sinal Cru: Registo 100, primeiros 10 s

* Janela de 10 segundos (`fs · 10 = 3600 amostras`) no canal MLII (derivação modificada II).
* As linhas verticais marcam as anotações WFDB dos picos R fornecidas pelos cardiologistas.
* Observa se a oscilação de baixa frequência da linha de base e o ruído fino sobreposto ao traçado.

### 4.1 Ferramentas de Visualização e Análise Espectral

Para avaliar o impacto dos filtros FIR no sinal de ECG, precisamos de ferramentas robustas de visualização. Esta seção define duas funções principais que serão reutilizadas ao longo do notebook:

1. **`plot_ecg_signal`**: Focada puramente na morfologia do domínio do tempo, permitindo inspecionar distorções na onda QRS ou artefatos introduzidos.
2. **`plot_spectral_analysis`**: Um painel integrado contendo:
   - O sinal no domínio do tempo.
   - **Espectrograma (STFT)**: Para visualizar a densidade de energia transitória (útil para ver quando o ruído de rede de 60Hz ocorre).
   - **Densidade Espectral de Potência (PSD - Welch)**: Para avaliar globalmente a atenuação nas bandas de corte (ex: atenuação abaixo de 0.5 Hz e acima de 40 Hz).

In [ ]:
def plot_ecg_signal(
    t: np.ndarray, 
    x: np.ndarray, 
    ann_samples: np.ndarray = None, 
    fs: float = None, 
    title: str = "Sinal ECG",
    ax: matplotlib.axes.Axes = None
):
    """
    Plota o sinal de ECG no domínio do tempo com anotações de picos R.
    Pode operar standalone ou ser injetada em um painel maior usando o parâmetro 'ax'.
    """
    is_standalone = False
    if ax is None:
        fig, ax = plt.subplots(figsize=(12, 4))
        is_standalone = True

    ax.plot(t, x, color="#1f77b4", linewidth=0.9, label="ECG (MLII)")
    
    if ann_samples is not None and fs is not None:
        y_margin = np.ptp(x) * 0.05 
        y_marker = np.max(x) + y_margin
        
        ax.scatter(
            ann_samples / fs, 
            np.full(len(ann_samples), y_marker),
            marker="v", color="#d62728", s=25, label="Picos R", zorder=3
        )
        ax.set_ylim(bottom=np.min(x) - y_margin, top=y_marker + y_margin*2)
            
    ax.set_xlabel("Tempo (s)")
    ax.set_ylabel("Amplitude (mV)")
    ax.set_title(title, fontweight='bold' if not is_standalone else 'normal')
    ax.grid(True, alpha=0.3)
    ax.set_xlim(t[0], t[-1])
    ax.legend(loc="upper right", fontsize=9)
    
    if is_standalone:
        fig.tight_layout()
        plt.show()

def plot_spectral_analysis(
    x: np.ndarray, 
    fs: float, 
    ann_samples: np.ndarray = None, 
    title_prefix: str = "Sinal", 
    max_freq: float = 80.0,
    hp_cut: float = 0.5,
    lp_cut: float = 40.0,
    pli_band: tuple[float, float] = (59.0, 61.0)
):
    """
    Gera um painel integrado com domínio do tempo, Espectrograma e PSD.
    Utiliza GridSpec para garantir o alinhamento perfeito do eixo X temporal.
    """
    t = np.arange(len(x)) / fs
    
    fig = plt.figure(figsize=(12, 9))
    
    # Criando um grid 3x2:
    # A 1ª coluna (larga) é para os gráficos. A 2ª coluna (fina) é apenas para a colorbar.
    gs = fig.add_gridspec(
        3, 2, 
        width_ratios=[1, 0.015], # 2ª coluna bem estreita
        height_ratios=[1, 1.2, 1.2], 
        wspace=0.03, # Espaço mínimo entre o gráfico e a colorbar
        hspace=0.45  # Espaço vertical para não sobrepor os títulos e labels
    )
    
    # 1. DOMÍNIO DO TEMPO (Ocupa apenas a coluna 0)
    ax1 = fig.add_subplot(gs[0, 0])
    plot_ecg_signal(
        t=t, 
        x=x, 
        ann_samples=ann_samples, 
        fs=fs, 
        title=f"{title_prefix} - Domínio do Tempo", 
        ax=ax1
    )
    # Removemos o rótulo do eixo X aqui para ficar mais limpo, já que o espectrograma tem a mesma escala
    ax1.set_xlabel("") 
    
    # 2. ESPECTROGRAMA (STFT) - Compartilha o eixo X com o ax1
    ax2 = fig.add_subplot(gs[1, 0], sharex=ax1)
    nperseg_stft = int(fs * 0.5) 
    noverlap = int(nperseg_stft * 0.9)
    f_stft, t_stft, Sxx = sp_signal.spectrogram(x, fs, nperseg=nperseg_stft, noverlap=noverlap)
    
    Sxx_dB = 10 * np.log10(Sxx + 1e-10)
    img = ax2.pcolormesh(t_stft, f_stft, Sxx_dB, shading='gouraud', cmap='magma', vmin=np.percentile(Sxx_dB, 5))
    ax2.set_title(f"{title_prefix} - Espectrograma (STFT)", fontweight='bold')
    ax2.set_ylabel("Frequência (Hz)")
    ax2.set_xlabel("Tempo (s)")
    ax2.set_ylim(0, max_freq)
    
    # Eixo dedicado APENAS para a colorbar, mantendo a largura do ax2 intacta
    cax = fig.add_subplot(gs[1, 1])
    cbar = fig.colorbar(img, cax=cax)
    cbar.set_label("Potência (dB)", rotation=270, labelpad=15)
    
    # 3. DENSIDADE ESPECTRAL DE POTÊNCIA (WELCH) - Domínio da Frequência
    ax3 = fig.add_subplot(gs[2, 0])
    nperseg_welch = int(2.0 * fs) 
    f_welch, Pxx_welch = sp_signal.welch(x, fs, nperseg=nperseg_welch)
    
    ax3.semilogy(f_welch, Pxx_welch, color="#ff7f0e", lw=1.2)
    
    ax3.axvspan(0, hp_cut, color='gray', alpha=0.2, label=f'Baseline (< {hp_cut} Hz)')
    ax3.axvspan(pli_band[0], pli_band[1], color='red', alpha=0.15, label=f'Rede ({pli_band[0]}-{pli_band[1]} Hz)')
    ax3.axvline(lp_cut, color='black', linestyle='--', alpha=0.7, label=f'Corte LP ({lp_cut} Hz)')
    
    ax3.set_title(f"{title_prefix} - Densidade Espectral de Potência (Welch)", fontweight='bold')
    ax3.set_xlabel("Frequência (Hz)")
    ax3.set_ylabel("Densidade ($mV^2/Hz$)")
    ax3.set_xlim(0, max_freq)
    ax3.grid(True, alpha=0.3, which='both')
    ax3.legend(fontsize=9, loc='upper right')
    
    # Ao invés de tight_layout que pode bagunçar o gridspec, alinhamos manualmente se necessário
    # fig.tight_layout() # Opcional agora, pois o GridSpec já define bons espaços.
    plt.show()

### 4.2 Visualizando o Sinal

In [ ]:
fs = float(record.fs)
window = int(fs * 10)
t = np.arange(window) / fs

ch = 0  # MLII
x_raw = record.p_signal[:window, ch]

# anotacoes corrigidas
samples = np.array(ann.sample)
symbols = np.array(ann.symbol)

# 1. Mascara de tempo: apenas anotacoes dentro da janela de 10s
mask_time = samples < window

# 2. Mascara de tipo: exclui metadados ('+', '~', '"') e mantem batimentos reais.
# 'N' representa batimento normal. Adicionamos outros comuns caso mude de registro.
valid_beats = ['N', 'L', 'R', 'B', 'A', 'a', 'J', 'S', 'V', 'r', 'F', 'e', 'j', 'n', 'E', '/', 'f', 'Q', '?']
mask_type = np.isin(symbols, valid_beats)

# Aplica as duas restricoes simultaneamente
ann_in_window = samples[mask_time & mask_type]

# primeiro inspecionando apenas a morfologia no tempo
plot_ecg_signal(
    t=t, 
    x=x_raw, 
    ann_samples=ann_in_window, 
    fs=fs, 
    title=f"MIT BIH {RECORD_NAME}: Sinal Cru (MLII) - Primeiros 10s"
)

# dps inspecionando o perfil espectral completo
plot_spectral_analysis(
    x=x_raw,
    fs=fs,
    ann_samples=ann_in_window,
    title_prefix=f"MIT BIH {RECORD_NAME} Sinal Cru",
    max_freq=80.0
)

## 5. Estratégia de Filtragem: FIR de Fase Linear

A Fase 1 utiliza uma cascata de três filtros FIR projetados sobre `fs = 360 Hz`:

1. **Passa-alta (0.5 Hz):** Voltado à remoção do baseline wander.
2. **Rejeita-faixa (59-61 Hz):** Voltado à interferência da rede elétrica (PLI).
3. **Passa-baixa (40 Hz):** Voltado à atenuação de ruído muscular de alta frequência.

São comparadas duas técnicas de projeto:
* **Janela de Hamming:** Método clássico, simples e robusto, com ordem aproximada de `3.3 · fs / Δf`.
* **Parks-McClellan (Remez):** Projeto equiripple ótimo no sentido de Chebyshev, alcançando ordens menores para uma mesma especificação.

In [ ]:
# Pipeline FIR projetado pela janela de Hamming
h_hp_ham = design_highpass(fs=fs, cutoff_hz=0.5, transition_hz=0.5)
h_bs_ham = design_bandstop(fs=fs, low_hz=59.0, high_hz=61.0, transition_hz=1.0)
h_lp_ham = design_lowpass(fs=fs, cutoff_hz=40.0, transition_hz=8.0)

# Pipeline FIR projetado por Parks-McClellan (equiripple)
h_hp_pm = design_highpass_remez(fs=fs, cutoff_hz=0.5, transition_hz=0.5)
h_bs_pm = design_bandstop_remez(fs=fs, low_hz=59.0, high_hz=61.0, transition_hz=1.0)
h_lp_pm = design_lowpass_remez(fs=fs, cutoff_hz=40.0, transition_hz=8.0)

print("Ordem dos Filtros Projetados (N):")
print(f"Hamming         | HP: {len(h_hp_ham):4d} | BS: {len(h_bs_ham):4d} | LP: {len(h_lp_ham):4d}")
print(f"Parks-McClellan | HP: {len(h_hp_pm):4d} | BS: {len(h_bs_pm):4d} | LP: {len(h_lp_pm):4d}")

# Carregando o sinal completo para aplicação filtfilt (evita transientes nas bordas da janela)
x_full = record.p_signal[:, ch]

### 5.1 Evolução Espectral: Aplicação em Cascata

Para compreender o impacto acústico/espectral de cada estágio, aplicaremos os filtros da cascata sequencialmente (utilizando o design Parks-McClellan como base) e avaliaremos as mudanças através do espectrograma e do PSD.

In [ ]:
# PASSO 1: Aplicação do filtro Passa-Alta (Remoção do Baseline Wander)
xp_step1 = apply_filtfilt(h_hp_pm, x_full)

# Plotando os primeiros 10 segundos
plot_spectral_analysis(
    x=xp_step1[:window], 
    fs=fs, 
    ann_samples=ann_in_window, 
    title_prefix="1. Após Filtro Passa-Alta (0.5 Hz)"
)
# Nota Analítica: Observe no gráfico de Welch (laranja) como a potência cai drasticamente 
# na área cinza (próxima a 0 Hz) em comparação ao sinal cru.

In [ ]:
# PASSO 2: Aplicação do filtro Rejeita-Faixa (Notch 60 Hz) sobre o sinal anterior
xp_step2 = apply_filtfilt(h_bs_pm, xp_step1)

plot_spectral_analysis(
    x=xp_step2[:window], 
    fs=fs, 
    ann_samples=ann_in_window, 
    title_prefix="2. Após Filtro Rejeita-Faixa (60 Hz)"
)

# PASSO 3: Aplicação do filtro Passa-Baixa (Corte 40 Hz) sobre o sinal anterior
x_filt_pm_full = apply_filtfilt(h_lp_pm, xp_step2)
x_filt_pm = x_filt_pm_full[:window] # Separando a janela de 10s para comparações finais

plot_spectral_analysis(
    x=x_filt_pm, 
    fs=fs, 
    ann_samples=ann_in_window, 
    title_prefix="3. Após Filtro Passa-Baixa (40 Hz) - Pipeline Completo"
)
# Nota Analítica: Note que o PSD e o espectrograma ficam limpos acima da linha de 40 Hz.

### 6. Comparação Analítica: Hamming vs Parks-McClellan

Com os sinais totalmente filtrados, precisamos comparar as abordagens de design empírico (Hamming) contra a otimização equiripple (Parks-McClellan). A comparação visualiza a superposição no domínio do tempo e na Densidade Espectral de Potência.

In [ ]:
def plot_pipeline_comparison(t, x_raw, x_ham, x_pm, fs):
    """
    Sobrepõe o sinal cru e as duas abordagens de filtragem 
    no domínio do tempo e no domínio da frequência (PSD).
    """
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), gridspec_kw={'height_ratios': [1, 1.2]})

    # ----------------------------------------------------
    # DOMÍNIO DO TEMPO
    # ----------------------------------------------------
    ax1.plot(t, x_raw, color="gray", alpha=0.3, lw=1.0, label="Original (Cru)")
    ax1.plot(t, x_ham, color="tab:orange", lw=1.2, alpha=0.8, label="Cascata Hamming")
    # Usando tracejado para PM para podermos ver se ele se sobrepõe perfeitamente ao Hamming
    ax1.plot(t, x_pm, color="tab:green", lw=1.2, alpha=0.9, linestyle="--", label="Cascata Parks-McClellan")
    
    ax1.set_title("Comparação de Pipelines no Domínio do Tempo", fontweight="bold")
    ax1.set_xlabel("Tempo (s)")
    ax1.set_ylabel("Amplitude (mV)")
    ax1.set_xlim(t[0], t[-1])
    ax1.legend(loc="upper right")
    ax1.grid(True, alpha=0.3)

    # ----------------------------------------------------
    # DOMÍNIO DA FREQUÊNCIA (WELCH)
    # ----------------------------------------------------
    nperseg_welch = int(2.0 * fs)
    f_raw, Pxx_raw = sp_signal.welch(x_raw, fs, nperseg=nperseg_welch)
    f_ham, Pxx_ham = sp_signal.welch(x_ham, fs, nperseg=nperseg_welch)
    f_pm, Pxx_pm  = sp_signal.welch(x_pm, fs, nperseg=nperseg_welch)

    ax2.semilogy(f_raw, Pxx_raw, color="gray", alpha=0.3, lw=1.5, label="Original (Cru)")
    ax2.semilogy(f_ham, Pxx_ham, color="tab:orange", lw=1.2, alpha=0.8, label="Cascata Hamming")
    ax2.semilogy(f_pm, Pxx_pm, color="tab:green", lw=1.2, alpha=0.9, linestyle="--", label="Cascata Parks-McClellan")
    
    # Demarcações de corte globais
    ax2.axvline(0.5, color='gray', linestyle=':', alpha=0.7)
    ax2.axvline(40.0, color='black', linestyle=':', alpha=0.7)
    
    ax2.set_title("Eficiência do Roll-off (Densidade Espectral de Potência)", fontweight="bold")
    ax2.set_xlabel("Frequência (Hz)")
    ax2.set_ylabel("Densidade ($mV^2/Hz$)")
    ax2.set_xlim(0, 80)
    ax2.legend(loc="upper right")
    ax2.grid(True, alpha=0.3, which="both")

    fig.tight_layout()
    plt.show()

xh = apply_filtfilt(h_hp_ham, x_full)
xh = apply_filtfilt(h_bs_ham, xh)
x_filt_ham_full = apply_filtfilt(h_lp_ham, xh)
x_filt_ham = x_filt_ham_full[:window]

# Plotando a comparação final
plot_pipeline_comparison(t, x_raw, x_filt_ham, x_filt_pm, fs)

## 7. Análise de Padrões Locais com Filtro de Gabor 1D

Nesta etapa, exploramos a utilizacao de um filtro de Gabor 1D para atuar como um detector de padroes (template matching). O objetivo e sintonizar o filtro para ressoar com as transicoes rapidas do complexo QRS, isolando-o de ruidos e ondas lentas (como P e T).

### 7.1 Fundamentos Teoricos e Comportamento do Kernel

Matematicamente, o filtro de Gabor 1D e definido pelo produto de uma onda senoidal (portadora) e uma envoltoria Gaussiana (janela limitadora):

$$g(t; f_0, \sigma, \phi) = \exp\left(-\frac{t^2}{2\sigma^2}\right) \cos(2\pi f_0 t + \phi)$$

A escolha dos parametros determina exatamente qual morfologia o filtro ira realcar:

* **Frequencia central ($f_0$):** Controla a taxa de oscilacao da senoidal. Deve ser ajustada para a banda de energia dominante do evento alvo (para o complexo QRS, tipicamente entre 10 e 20 Hz).
* **Largura da Gaussiana ($\sigma$):** Define a "abertura" da janela no tempo. Um $\sigma$ pequeno cria um filtro altamente localizado, ignorando oscilacoes vizinhas. Um $\sigma$ grande permite que mais ciclos da senoidal atuem, tornando o filtro mais sensivel a eventos de longa duracao.
* **Truncamento temporal ($3\sigma$):** A funcao Gaussiana se estende ao infinito. No entanto, o intervalo de $[-3\sigma, +3\sigma]$ engloba aproximadamente 99.7% da area da curva. Portanto, truncar o vetor de tempo em $t_{max} = 3\sigma$ garante que o decaimento do kernel seja modelado por completo, otimizando o custo computacional da convolucao sem perda de precisao.

As visualizacoes a seguir demonstram o impacto isolado de cada parametro na forma do kernel.

In [ ]:
# ====================================================================
# Visualizacao 1: Impacto da Frequencia Central (f0)
# ====================================================================
# Mantemos a largura da envoltoria constante para observar como a
# onda interna "acelera" de acordo com a frequencia sintonizada.

sigma_fixo = 0.04
# Testando frequencias: 5 Hz (onda lenta), 15 Hz (QRS tipico), 30 Hz (ruido rapido)
f0_valores = [5.0, 15.0, 30.0] 

# O vetor de tempo obedece a regra empirica de t_max = 3 * sigma
t_gabor1 = np.arange(-3 * sigma_fixo, 3 * sigma_fixo, 1 / fs)

fig, ax = plt.subplots(figsize=(10, 3.5))

for f0_val in f0_valores:
    kernel_f0 = np.exp(-(t_gabor1**2) / (2 * sigma_fixo**2)) * np.cos(2 * np.pi * f0_val * t_gabor1)
    ax.plot(t_gabor1, kernel_f0, label=fr"$f_0$ = {f0_val} Hz", lw=1.5)

ax.set_title(fr"Variacao da Frequencia Central ($f_0$) com $\sigma$ fixo ({sigma_fixo} s)", fontweight="bold")
ax.set_xlabel("Tempo (s)")
ax.set_ylabel("Amplitude")
ax.grid(True, alpha=0.3)
ax.legend(loc="upper right")

fig.tight_layout()
plt.show()

In [ ]:
# ====================================================================
# Visualizacao 2: Impacto da Largura da Envoltoria Gaussiana (sigma)
# ====================================================================
# Mantemos a oscilacao constante para observar como a Gaussiana 
# restringe (ou expande) o tempo de atuacao do filtro.

f0_fixo = 15.0
# Testando duracoes: 0.015s (muito restrito), 0.04s (ideal), 0.10s (muito amplo)
sigma_valores = [0.015, 0.04, 0.10] 

fig, ax = plt.subplots(figsize=(10, 3.5))

# Para plotar juntos no mesmo eixo X, usamos o tempo baseado no MAIOR sigma
t_max_all = 3 * max(sigma_valores)
t_gabor2 = np.arange(-t_max_all, t_max_all, 1 / fs)

for sig_val in sigma_valores:
    # A equacao forca a amplitude a zero nas bordas naturalmente para sigmas menores
    kernel_sig = np.exp(-(t_gabor2**2) / (2 * sig_val**2)) * np.cos(2 * np.pi * f0_fixo * t_gabor2)
    ax.plot(t_gabor2, kernel_sig, label=fr"$\sigma$ = {sig_val} s", lw=1.5)

ax.set_title(fr"Variação da Envoltória Gaussiana ($\sigma$) com $f_0$ fixo ({f0_fixo} Hz)", fontweight="bold")
ax.set_xlabel("Tempo (s)")
ax.set_ylabel("Amplitude")
ax.grid(True, alpha=0.3)
ax.legend(loc="upper right")

fig.tight_layout()
plt.show()

### 7.2 Implementacao e Convolucao com o Sinal de ECG

Com a intuicao geometrica estabelecida, definimos as funcoes principais para a geracao do kernel e para a convolucao contra o sinal de ECG original. Avaliaremos a energia da resposta (amplitude ao quadrado) como indicativo de ressonancia com os picos R.

In [ ]:
def generate_gabor_1d(fs: float, f0: float, sigma: float, phi: float = 0.0) -> tuple[np.ndarray, np.ndarray]:
    """
    Gera kernel de Gabor 1D.
    Duracao do kernel definida como 6*sigma.
    """
    t_max = 3 * sigma
    t = np.arange(-t_max, t_max, 1/fs)
    
    # Equacao (3)
    g = np.exp(-(t**2) / (2 * sigma**2)) * np.cos(2 * np.pi * f0 * t + phi)
    
    return t, g

def plot_gabor_experiment(
    t_signal: np.ndarray, 
    x_signal: np.ndarray, 
    fs: float, 
    ann_samples: np.ndarray, 
    f0: float, 
    sigma: float, 
    phi: float = 0.0
):
    """
    Aplica filtro de Gabor e plota resultados.
    """
    t_kernel, gabor_kernel = generate_gabor_1d(fs, f0, sigma, phi)
    
    # Convolucao com mesmo tamanho para alinhamento temporal
    response = np.convolve(x_signal, gabor_kernel, mode='same')
    energy_response = response**2

    fig = plt.figure(figsize=(12, 8))
    gs = fig.add_gridspec(3, 2, height_ratios=[1, 1.5, 1.5], width_ratios=[1, 4])
    
    # Painel A: Kernel
    ax_kernel = fig.add_subplot(gs[0, 0])
    ax_kernel.plot(t_kernel, gabor_kernel, color='purple', lw=1.5)
    ax_kernel.set_title(fr"Kernel: $f_0$={f0}Hz, $\sigma$={sigma}s", fontsize=9)
    ax_kernel.grid(True, alpha=0.3)
    
    ax_empty = fig.add_subplot(gs[0, 1])
    ax_empty.axis('off')
    
    # Painel B: Sinal original e marcacoes
    ax_ecg = fig.add_subplot(gs[1, :])
    ax_ecg.plot(t_signal, x_signal, color='tab:green', lw=1.0, label="ECG Pre-filtrado")
    
    y_marker = np.max(x_signal) * 1.1
    ax_ecg.scatter(ann_samples / fs, np.full(len(ann_samples), y_marker),
                   marker="v", color="tab:red", s=30, label="Picos R", zorder=3)
    
    ax_ecg.set_ylabel("Amplitude (mV)")
    ax_ecg.set_title("Sinal Base e Anotacoes Originais", fontweight='bold')
    ax_ecg.grid(True, alpha=0.3)
    ax_ecg.legend(loc='upper right')
    ax_ecg.set_xlim(t_signal[0], t_signal[-1])
    
    # Painel C: Resposta da convolucao
    ax_resp = fig.add_subplot(gs[2, :], sharex=ax_ecg)
    ax_resp.plot(t_signal, response, color='tab:blue', alpha=0.6, lw=1.0, label="Resposta")
    ax_resp.plot(t_signal, energy_response / np.max(energy_response) * np.max(response), 
                 color='tab:orange', lw=1.5, label="Energia Normalizada ($y^2$)")
    
    for s in ann_samples:
        ax_resp.axvline(s / fs, color='tab:red', linestyle='--', alpha=0.4, lw=0.8)
        
    ax_resp.set_xlabel("Tempo (s)")
    ax_resp.set_ylabel("Amplitude / Energia")
    ax_resp.set_title("Resposta do Filtro de Gabor", fontweight='bold')
    ax_resp.grid(True, alpha=0.3)
    ax_resp.legend(loc='upper right')
    
    fig.tight_layout()
    plt.show()

In [ ]:
# f0: Frequência central. O QRS tem sua maior energia entre 10 e 20 Hz.
f0_test = 15.0

# sigma: Controla a "largura" da envoltória Gaussiana simples. 
# Um QRS dura em média 0.08 a 0.12s. 
# sigma = 0.03 a 0.05 foca bem nessa janela.
sigma_test = 0.01

# Usando o sinal CRU (x_raw) para evitar os artefatos (ringing/Gibbs) 
# introduzidos pelo filtro de Parks-McClellan na etapa anterior.
plot_gabor_experiment(
    t_signal=t, 
    x_signal=x_raw,       # <- Mudamos aqui para o sinal original!
    fs=fs, 
    ann_samples=ann_in_window, 
    f0=f0_test, 
    sigma=sigma_test      # A fase (phi) foi omitida, assumindo o padrão 0.0 (simétrico)
)

### 7.1 Avaliacao do Alinhamento Temporal (Spread)

Para validar a precisao do filtro de Gabor como detector de QRS, calculamos a diferenca temporal entre as anotacoes originais do MIT-BIH e os picos de energia produzidos pela convolucao.

O erro e medido em milissegundos (ms). Erros proximos de zero indicam que a configuracao atual (f0 e sigma) entra em ressonancia perfeita com o centro do complexo QRS.

In [ ]:
def evaluate_gabor_alignment(
    x_signal: np.ndarray, 
    fs: float, 
    ann_samples: np.ndarray, 
    f0: float, 
    sigma: float, 
    phi: float = 0.0,
    search_window_sec: float = 0.1
) -> pd.DataFrame:
    """
    Compara a posicao dos picos R anotados com os picos de energia do Gabor.
    Retorna tabela Pandas e plota o desvio (spread).
    """
    # Gera kernel e calcula energia
    _, gabor_kernel = generate_gabor_1d(fs, f0, sigma, phi)
    response = np.convolve(x_signal, gabor_kernel, mode='same')
    energy = response**2
    
    window_samples = int(search_window_sec * fs)
    records = []
    
    for i, idx in enumerate(ann_samples):
        # Janela de busca restrita ao redor da anotacao
        start = max(0, idx - window_samples)
        end = min(len(energy), idx + window_samples)
        
        # Indice local do pico de energia
        gabor_idx = start + np.argmax(energy[start:end])
        
        # Erro de alinhamento em ms
        err_ms = ((gabor_idx - idx) / fs) * 1000.0
        
        records.append({
            "Batimento": i + 1,
            "Ref (s)": round(idx / fs, 3),
            "Gabor (s)": round(gabor_idx / fs, 3),
            "Erro (ms)": round(err_ms, 2)
        })
        
    df = pd.DataFrame(records)
    
    # Visualizacao do desvio
    fig, ax = plt.subplots(figsize=(10, 3.5))
    
    ax.stem(df["Batimento"], df["Erro (ms)"], basefmt="k-", linefmt="tab:red", markerfmt="ro")
    
    # Limites visuais de tolerancia padrao na literatura (+- 20ms)
    ax.axhline(20, color='gray', linestyle='--', alpha=0.5, label="Tolerancia +20ms")
    ax.axhline(-20, color='gray', linestyle='--', alpha=0.5, label="Tolerancia -20ms")
    
    ax.set_title("Desvio Temporal: Referencia vs Energia do Gabor", fontweight="bold")
    ax.set_xlabel("Indice do Batimento")
    ax.set_ylabel("Erro (ms)")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="upper right")
    
    fig.tight_layout()
    plt.show()
    
    mae = df["Erro (ms)"].abs().mean()
    print(f"Erro Absoluto Medio (MAE): {mae:.2f} ms\n")
    
    return df

# Executa a avaliacao com os parametros testados
df_results = evaluate_gabor_alignment(
    x_signal=x_raw, 
    fs=fs, 
    ann_samples=ann_in_window, 
    f0=f0_test,      # Variaveis definidas na celula anterior
    sigma=sigma_test
)

# Exibe a tabela Pandas formatada
display(df_results)

## Lab 12/05/2026
- Implementar análise do Gabor 1D variando seus parâmetros
- Analisar o funcionamento do Gabor 1D no domínio da frequência